# UCB1: Optimism in the Face of Uncertainty

**Goal:** Implement UCB1 from scratch and compare it to epsilon-greedy.

**Time:** 15 minutes

**You'll learn:**
- How UCB uses confidence bounds for exploration
- Why UCB achieves logarithmic regret vs ε-greedy's T^(2/3)
- When UCB beats ε-greedy and when it doesn't

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Environment: Same Commodity Bandit

In [ ]:
class CommodityBandit:
    def __init__(self, means, stds):
        self.means = np.array(means)
        self.stds = np.array(stds)
        self.k = len(means)
        self.best_arm = np.argmax(means)
        self.best_mean = np.max(means)
    
    def pull(self, arm):
        reward = np.random.normal(self.means[arm], self.stds[arm])
        return np.clip(reward, 0, 1)
    
    def get_regret(self, arm):
        return self.best_mean - self.means[arm]

sector_names = ['Energy', 'Metals', 'Agriculture', 'Livestock', 'Softs']
bandit = CommodityBandit(
    means=[0.3, 0.5, 0.8, 0.4, 0.2],
    stds=[0.5, 0.3, 0.2, 0.4, 0.6]
)

print(f"Environment: 5 commodity sectors")
print(f"Best arm: {sector_names[bandit.best_arm]} (μ={bandit.best_mean})")

## Implement UCB1 from Scratch

**Algorithm:**
```
UCB(a) = Q̂(a) + c · √(ln(t) / N(a))

Select: argmax_a UCB(a)
```

**Intuition:** Pick the arm that could plausibly be the best (optimistic upper bound).

In [ ]:
class UCB1:
    def __init__(self, k_arms, c=np.sqrt(2)):
        self.k = k_arms
        self.c = c
        self.q_estimates = np.zeros(k_arms)
        self.action_counts = np.zeros(k_arms)
        self.t = 0  # Total time steps
    
    def select_action(self):
        self.t += 1
        
        # First K pulls: try each arm once (avoid division by zero)
        if self.t <= self.k:
            return self.t - 1
        
        # Compute UCB values
        ucb_values = self.q_estimates + self.c * np.sqrt(
            np.log(self.t) / (self.action_counts + 1e-10)
        )
        
        return np.argmax(ucb_values)
    
    def update(self, action, reward):
        self.action_counts[action] += 1
        n = self.action_counts[action]
        self.q_estimates[action] += (reward - self.q_estimates[action]) / n
    
    def get_ucb_values(self):
        """Get current UCB values for visualization"""
        if self.t <= self.k:
            return np.zeros(self.k)
        return self.q_estimates + self.c * np.sqrt(
            np.log(self.t) / (self.action_counts + 1e-10)
        )
    
    def get_confidence_radius(self):
        """Get the confidence bonus for each arm"""
        if self.t <= self.k:
            return np.zeros(self.k)
        return self.c * np.sqrt(np.log(self.t) / (self.action_counts + 1e-10))

ucb_agent = UCB1(k_arms=5, c=np.sqrt(2))
print(f"UCB1 agent initialized with c={ucb_agent.c:.3f}")

## Compare UCB1 vs Epsilon-Greedy

Run both algorithms on the same problem and track regret.

In [ ]:
# Epsilon-greedy for comparison
class EpsilonGreedy:
    def __init__(self, k_arms, epsilon=0.1):
        self.k = k_arms
        self.epsilon = epsilon
        self.q_estimates = np.zeros(k_arms)
        self.action_counts = np.zeros(k_arms)
    
    def select_action(self):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.k)
        else:
            return np.argmax(self.q_estimates)
    
    def update(self, action, reward):
        self.action_counts[action] += 1
        n = self.action_counts[action]
        self.q_estimates[action] += (reward - self.q_estimates[action]) / n

# Run both algorithms
T = 10000
ucb = UCB1(k_arms=5)
egreedy = EpsilonGreedy(k_arms=5, epsilon=0.1)

regrets_ucb = []
regrets_egreedy = []

for t in range(T):
    # UCB1
    a1 = ucb.select_action()
    r1 = bandit.pull(a1)
    ucb.update(a1, r1)
    regrets_ucb.append(bandit.get_regret(a1))
    
    # Epsilon-greedy
    a2 = egreedy.select_action()
    r2 = bandit.pull(a2)
    egreedy.update(a2, r2)
    regrets_egreedy.append(bandit.get_regret(a2))

print(f"Final regret:")
print(f"  UCB1:           {np.sum(regrets_ucb):.2f}")
print(f"  ε-greedy:       {np.sum(regrets_egreedy):.2f}")
print(f"  UCB improvement: {(1 - np.sum(regrets_ucb)/np.sum(regrets_egreedy))*100:.1f}%")

## Plot Regret Comparison

In [ ]:
plt.figure(figsize=(12, 5))

# Cumulative regret
plt.subplot(1, 2, 1)
plt.plot(np.cumsum(regrets_ucb), label='UCB1', linewidth=2.5, color='#2ecc71')
plt.plot(np.cumsum(regrets_egreedy), label='ε-greedy (ε=0.1)', linewidth=2.5, color='#e74c3c')
plt.xlabel('Time Step')
plt.ylabel('Cumulative Regret')
plt.title('Cumulative Regret: UCB1 vs ε-greedy')
plt.legend()
plt.grid(True, alpha=0.3)

# Pull counts
plt.subplot(1, 2, 2)
x = np.arange(5)
width = 0.35
plt.bar(x - width/2, ucb.action_counts, width, label='UCB1', alpha=0.8, color='#2ecc71')
plt.bar(x + width/2, egreedy.action_counts, width, label='ε-greedy', alpha=0.8, color='#e74c3c')
plt.xlabel('Arm')
plt.ylabel('Number of Pulls')
plt.title('Arm Selection Frequency')
plt.xticks(x, sector_names, rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nPull count comparison:")
print(f"{'Sector':12s}  {'UCB1':>6s}  {'ε-greedy':>9s}")
print("-" * 35)
for i, name in enumerate(sector_names):
    best = " ⭐" if i == bandit.best_arm else ""
    print(f"{name:12s}  {int(ucb.action_counts[i]):6d}  {int(egreedy.action_counts[i]):9d}{best}")

## Visualize Confidence Bounds Over Time

Let's see how the confidence bounds shrink as we pull each arm more.

In [ ]:
# Reset and track confidence bounds
ucb = UCB1(k_arms=5)
snapshots = [10, 50, 200, 1000, 5000, 10000]  # Time points to visualize
confidence_history = {t: [] for t in snapshots}
q_history = {t: [] for t in snapshots}
ucb_history = {t: [] for t in snapshots}

for t in range(max(snapshots)):
    action = ucb.select_action()
    reward = bandit.pull(action)
    ucb.update(action, reward)
    
    # Save snapshots
    if t + 1 in snapshots:
        confidence_history[t + 1] = ucb.get_confidence_radius()
        q_history[t + 1] = ucb.q_estimates.copy()
        ucb_history[t + 1] = ucb.get_ucb_values()

# Plot confidence bounds at different times
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, t in enumerate(snapshots):
    ax = axes[idx]
    
    q = q_history[t]
    conf = confidence_history[t]
    ucb_vals = ucb_history[t]
    
    # Plot Q̂ and confidence intervals
    x = np.arange(5)
    
    # Plot confidence intervals
    for i in range(5):
        ax.plot([i, i], [q[i] - conf[i], q[i] + conf[i]], 
               'k-', linewidth=6, alpha=0.3, solid_capstyle='round')
    
    # Plot Q̂ estimates
    ax.scatter(x, q, s=150, c='steelblue', zorder=10, edgecolors='black', linewidths=2)
    
    # Plot true means
    ax.scatter(x, bandit.means, s=100, c='red', marker='x', zorder=11, linewidths=3, label='True μ')
    
    # Highlight selected arm
    selected = np.argmax(ucb_vals)
    ax.scatter([selected], [ucb_vals[selected]], s=300, facecolors='none', 
              edgecolors='green', linewidths=3, zorder=12, label='Next choice')
    
    ax.set_ylim(-0.1, 1.3)
    ax.set_xticks(x)
    ax.set_xticklabels(sector_names, rotation=45, ha='right')
    ax.set_ylabel('Reward')
    ax.set_title(f't = {t}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(loc='upper right', fontsize=8)

plt.suptitle('Confidence Bounds Shrink Over Time (|—| = 95% confidence interval)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🎛️ Interactive: Adjust the Exploration Constant c

The constant c controls how optimistic UCB is:
- **c = √2:** Standard (theory-optimal)
- **c = 1:** Conservative (less exploration)
- **c = 2:** Aggressive (more exploration)

Try different values and see the impact on regret.

In [ ]:
# Experiment with different c values
c_values = [0.5, 1.0, np.sqrt(2), 2.0, 3.0]
results = []

for c in c_values:
    ucb = UCB1(k_arms=5, c=c)
    total_regret = 0
    
    for t in range(T):
        action = ucb.select_action()
        reward = bandit.pull(action)
        ucb.update(action, reward)
        total_regret += bandit.get_regret(action)
    
    results.append({
        'c': c,
        'regret': total_regret,
        'best_arm_pct': 100 * ucb.action_counts[bandit.best_arm] / T
    })

# Plot results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
c_vals = [r['c'] for r in results]
regrets = [r['regret'] for r in results]
plt.plot(c_vals, regrets, 'o-', linewidth=2, markersize=10, color='#3498db')
plt.axvline(np.sqrt(2), color='red', linestyle='--', alpha=0.7, label='c=√2 (standard)')
plt.xlabel('Exploration Constant (c)')
plt.ylabel('Total Regret')
plt.title('Impact of c on Regret')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
best_pcts = [r['best_arm_pct'] for r in results]
plt.plot(c_vals, best_pcts, 'o-', linewidth=2, markersize=10, color='#2ecc71')
plt.axvline(np.sqrt(2), color='red', linestyle='--', alpha=0.7, label='c=√2 (standard)')
plt.xlabel('Exploration Constant (c)')
plt.ylabel('% Pulls on Best Arm')
plt.title('Impact of c on Exploitation')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

print("\nExploration constant sensitivity:")
print(f"{'c':>6s}  {'Regret':>8s}  {'Best Arm %':>10s}")
print("-" * 30)
for r in results:
    marker = " ← standard" if abs(r['c'] - np.sqrt(2)) < 0.01 else ""
    print(f"{r['c']:>6.2f}  {r['regret']:>8.1f}  {r['best_arm_pct']:>10.1f}%{marker}")

## Compare Regret Growth Rates

Theory predicts:
- **UCB1:** O(ln T) regret (grows very slowly)
- **ε-greedy:** O(T^(2/3)) regret (grows faster)

Let's verify empirically by running for different time horizons.

In [ ]:
time_horizons = [100, 500, 1000, 5000, 10000, 50000]
ucb_regrets = []
egreedy_regrets = []

for T_test in time_horizons:
    # UCB1
    ucb = UCB1(k_arms=5)
    regret_ucb = 0
    for t in range(T_test):
        a = ucb.select_action()
        r = bandit.pull(a)
        ucb.update(a, r)
        regret_ucb += bandit.get_regret(a)
    ucb_regrets.append(regret_ucb)
    
    # ε-greedy
    eg = EpsilonGreedy(k_arms=5, epsilon=0.1)
    regret_eg = 0
    for t in range(T_test):
        a = eg.select_action()
        r = bandit.pull(a)
        eg.update(a, r)
        regret_eg += bandit.get_regret(a)
    egreedy_regrets.append(regret_eg)

# Plot on log-log scale
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(time_horizons, ucb_regrets, 'o-', linewidth=2.5, markersize=8, label='UCB1', color='#2ecc71')
plt.plot(time_horizons, egreedy_regrets, 'o-', linewidth=2.5, markersize=8, label='ε-greedy', color='#e74c3c')
plt.xlabel('Time Horizon (T)')
plt.ylabel('Total Regret')
plt.title('Regret vs Time Horizon')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.loglog(time_horizons, ucb_regrets, 'o-', linewidth=2.5, markersize=8, label='UCB1', color='#2ecc71')
plt.loglog(time_horizons, egreedy_regrets, 'o-', linewidth=2.5, markersize=8, label='ε-greedy', color='#e74c3c')

# Reference lines
T = np.array(time_horizons)
plt.loglog(T, 5 * np.log(T), 'k--', alpha=0.5, linewidth=2, label='O(ln T)')
plt.loglog(T, 0.3 * T**(2/3), 'k:', alpha=0.5, linewidth=2, label='O(T^(2/3))')

plt.xlabel('Time Horizon (T)')
plt.ylabel('Total Regret')
plt.title('Regret Growth Rate (log-log scale)')
plt.legend()
plt.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("\nRegret scaling:")
print(f"{'T':>8s}  {'UCB1':>8s}  {'ε-greedy':>10s}  {'UCB Advantage':>12s}")
print("-" * 50)
for i, T_val in enumerate(time_horizons):
    advantage = (1 - ucb_regrets[i]/egreedy_regrets[i]) * 100
    print(f"{T_val:8d}  {ucb_regrets[i]:8.1f}  {egreedy_regrets[i]:10.1f}  {advantage:11.1f}%")

## When UCB Fails: Heavy-Tailed Rewards

UCB1 assumes rewards are bounded. Let's see what happens with heavy-tailed distributions.

In [ ]:
class HeavyTailedBandit:
    """Bandit with Pareto-distributed rewards (heavy tails)"""
    def __init__(self, k_arms, alpha=2.0):
        self.k = k_arms
        self.alpha = alpha  # Shape parameter (smaller = heavier tail)
    
    def pull(self, arm):
        # Arm 2 has slightly better distribution
        if arm == 2:
            return np.random.pareto(self.alpha + 0.5) 
        else:
            return np.random.pareto(self.alpha)

# Test on heavy-tailed problem
ht_bandit = HeavyTailedBandit(k_arms=5)

ucb_ht = UCB1(k_arms=5)
eg_ht = EpsilonGreedy(k_arms=5, epsilon=0.1)

rewards_ucb_ht = []
rewards_eg_ht = []

for t in range(5000):
    # UCB
    a1 = ucb_ht.select_action()
    r1 = ht_bandit.pull(a1)
    ucb_ht.update(a1, r1)
    rewards_ucb_ht.append(r1)
    
    # ε-greedy
    a2 = eg_ht.select_action()
    r2 = ht_bandit.pull(a2)
    eg_ht.update(a2, r2)
    rewards_eg_ht.append(r2)

print("\nHeavy-tailed rewards (unbounded):")
print(f"  UCB1 total reward:      {np.sum(rewards_ucb_ht):.1f}")
print(f"  ε-greedy total reward:  {np.sum(rewards_eg_ht):.1f}")
print(f"\n  UCB1 arm selection:     {ucb_ht.action_counts}")
print(f"  ε-greedy arm selection: {eg_ht.action_counts}")
print(f"\n  Note: UCB can be unstable with heavy tails!")
print(f"  A single extreme outlier can dominate Q̂ forever.")

## Summary

### What We Built
- ✅ UCB1 implementation from scratch
- ✅ Head-to-head comparison with ε-greedy
- ✅ Visualization of shrinking confidence bounds
- ✅ Verified logarithmic regret scaling

### Key Takeaways

1. **UCB uses optimism for exploration**
   - Adds confidence bonus to each arm's estimate
   - Bonus shrinks as arm is pulled more (N ↑ → bonus ↓)
   - No wasted random exploration like ε-greedy

2. **UCB achieves better regret bounds**
   - O(ln T) gap-dependent regret
   - Beats ε-greedy's O(T^(2/3)) by a wide margin
   - Advantage grows with longer time horizons

3. **UCB is parameter-free (mostly)**
   - c = √2 works for almost everything
   - No need to tune like ε in ε-greedy

4. **UCB has limitations**
   - Assumes bounded rewards (breaks with heavy tails)
   - Assumes stationarity (bad for regime changes)
   - Must pull each arm once initially (slow cold start)

### UCB vs ε-Greedy: When to Use Each

**Choose UCB when:**
- Rewards are bounded (e.g., [0, 1])
- Environment is stationary
- Long time horizon (T > 1000)
- You want parameter-free algorithm

**Choose ε-greedy when:**
- Rewards are unbounded or heavy-tailed
- Environment is non-stationary
- Short time horizon
- You want simplicity and robustness

### What's Next?

**Next notebook:** Algorithm shootout—compare ε-greedy, UCB1, and softmax on real commodity data.

**Go deeper:** Read [guides/02_upper_confidence_bound.md](../guides/02_upper_confidence_bound.md) for the theory behind Hoeffding's inequality and regret analysis.